In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Configuration and paths
mac = 20

# ============================================================
# Variant class selection — change this to switch variant class
# Available: missense, missense_structured, missense_non_structured,
#   missense_disordered, missense_lip, intron,
#   enhancer_encode, promoter_encode, proximal_promoter, core_promoter
# ============================================================
variant_class = 'all_variants'

# selected_categories = ['protein_domains']
# selected_categories = ['plof_consequences']
selected_categories = ['clinvar']
# selected_categories = ['indel_lengths']
# selected_categories = ['coding_indels']
# selected_categories = ['non_coding_regions']

only_snps = False  # Whether to include only SNPs (exclude indels)

only_clinvar = False
exclude_clinvar = False  # Independent toggle: exclude ClinVar-annotated variants
ci_factor = 1.96  # For 95% confidence intervals
abs_phenotypes = False  # Whether to take absolute value of phenotypes (for direction-agnostic analysis)

eur_samples_path = 'PATH_TO_FILE'

# Load variant class configuration
variant_class_path = "PATH_TO_FILE"
with open(variant_class_path) as f:
    variant_class_config = yaml.safe_load(f)

vc = variant_class_config[variant_class]
vc_filters = vc['variant_filtering']

print(f"Variant class: {variant_class}")
print(f"  Filters: {vc_filters}")
print(f"  Exclude ClinVar: {exclude_clinvar}")

# Load annotation configuration
config_path = "PATH_TO_FILE" 

with open(config_path) as f:
    config = yaml.safe_load(f)
    
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

In [ ]:
# Subset Olink RVAT significant
LOCAL_DATA_DIR = "PATH_TO_FILE"

# olink_burden_test_file = "proteomics_prs_am_loftee_mac20_burden_regression_results.parquet"
# olink_burden_test_file = "proteomics_prs_loftee_mac20_burden_regression_results.parquet"
olink_burden_test_file = "proteomics_prs_df_loftee_mac20_burden_regression_results.parquet"
!dx download project-REDACTED:/processed_data/olink/blacklist/{olink_burden_test_file} -o {LOCAL_DATA_DIR}/{olink_burden_test_file}

olink_whitelist = (
    pl.read_parquet(f'{LOCAL_DATA_DIR}/{olink_burden_test_file}')
    .rename({'gene': 'region'})
    .filter((pl.col('padj')<=0.05) & (pl.col('wilcox_padj')<=0.05) )
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

# olink_correlations_file = "olink_all_mac20_lofteeHC_correlations.parquet"
olink_correlations_file = "olink_LOFTEE_correlations_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet"
!dx download project-REDACTED:/processed_data/REGENIE_results/{olink_correlations_file} -o {LOCAL_DATA_DIR}/{olink_correlations_file}

olink_corrs = (
    pl.read_parquet(f'{LOCAL_DATA_DIR}/{olink_correlations_file}')
    .filter(pl.col('correlation')>0)
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .drop_nans()
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

olink_whitelist = (
    olink_whitelist
    .join(olink_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    # .sort('loftee_corr_abs', descending=True)
    # .sort('pval_fdr')
    # .unique(subset=["region"], keep="first", maintain_order=True)
)

olink_whitelist

In [ ]:
RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "PATH_TO_FILE"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all_no_dup.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

# Build dynamic filters from variant_class.yaml
if vc_filters is not None:
    _dynamic_filters = [eval(f) for f in vc_filters]
else:
    _dynamic_filters = []

if exclude_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_null())
elif only_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_not_null())

if only_snps:
    _dynamic_filters.append((pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1))  
    
anno = (
    anno
    .with_columns(
        is_ins = pl.col('ref').str.len_chars() < pl.col('alt').str.len_chars(),
        is_del = pl.col('ref').str.len_chars() > pl.col('alt').str.len_chars(),
        encode_any_tf = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_tf', 'encode_ca-tf']),
        encode_enhancer = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pels', 'encode_dels']),
        encode_promoter = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pls', 'encode_ca-h3k4me3']),
        proximal_promoter = pl.col('dist2tssv39').abs() <= 500,
        core_promoter = pl.col('dist2tssv39').abs() <= 50,
    )
    .filter(
        # Always-applied filters
        (pl.col('region').is_in(olink_whitelist['region'].unique())),
        (pl.col('variant_length') <= 50),

        # Dynamic filters from variant_class.yaml + exclude_clinvar
        *_dynamic_filters,
    )
    .with_columns(
        ((pl.col('variant_length')==2) & (pl.col('is_del')==True)).alias('1bp_del'),
        ((pl.col('variant_length')==2) & (pl.col('is_ins')==True)).alias('1bp_ins'),

        ((pl.col('variant_length')>2) & (pl.col('variant_length')<=6) & (pl.col('is_del')==True)).alias('2_5bp_del'),
        ((pl.col('variant_length')>2) & (pl.col('variant_length')<=6) & (pl.col('is_ins')==True)).alias('2_5bp_ins'),

        ((pl.col('variant_length')>6) & (pl.col('is_del')==True)).alias('gt_5bp_del'),
        ((pl.col('variant_length')>6) & (pl.col('is_ins')==True)).alias('gt_5bp_ins'),

        inframe_deletion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_del')==True), #SNPs have length 1
        inframe_insertion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_ins')==True),

        clinvar_patho = pl.col('clinical_significance').str.contains('(?i)Pathogenic').fill_null(False),
        clinvar_likely_patho = pl.col('clinical_significance').str.contains('(?i)Likely_pathogenic').fill_null(False),
        clinvar_benign = pl.col('clinical_significance').str.contains('(?i)Benign').fill_null(False),
        clinvar_likely_benign = pl.col('clinical_significance').str.contains('(?i)Likely_benign').fill_null(False),

        non_ted_domain = pl.col('ted_domain') == False,
    )
)

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

In [ ]:
anno.sum()

In [ ]:
melted_anno = (
    anno.lazy()

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )

    .filter(pl.col('annotation_score')==1)

    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df.select(["annotation", "category", "annotation_dir"]).lazy(),
        on="annotation",
        how="left"
    )
    .filter(pl.col('category').is_in(selected_categories))
    .unique()

    .with_columns(
        # Changed because we have binary annotations
        annotation_score_dircor = (
            pl.when(pl.col("annotation_dir")!=1)
            .then((pl.col('annotation_score')-1).abs())
            .otherwise(pl.col('annotation_score'))
        )
    )

    .collect(engine='streaming')
)

melted_anno

In [ ]:
melted_anno['annotation'].value_counts(sort=True)

In [ ]:
RAP_APPV_DIR = "project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "PATH_TO_FILE"

# APPV_FILE = "cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet"
APPV_FILE = "olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
olink_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Small APPV file has no 'region' column — region is added via id_region join in the computation cell
olink_appv = (
    olink_appv
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .with_columns(
        mean_pheno_value = pl.when(abs_phenotypes)
        .then(pl.col('mean_pheno_value').abs())
        .otherwise(pl.col('mean_pheno_value'))
    )
    .select(['id', 'phenotype', 'mean_pheno_value', 'n_individuals'])
)

In [ ]:
all_pheno_df = (
    # A. Use a semi join for efficient filtering (more memory-safe than 'is_in')
    olink_appv
    # B. Join the other two dataframes
    .join(
        melted_anno.lazy(),
        on="id", 
        how="inner"
    )
    .join(
        olink_whitelist.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )
    .with_columns(
        mean_pheno_value_dircor = pl.col('mean_pheno_value')# *pl.col('loftee_corr_dir')
    )
    .group_by(["annotation", "region", "phenotype"])
    .agg(
        n_vars = pl.col('id').n_unique(),
        mean_pheno_assoc = pl.col('mean_pheno_value_dircor').mean(),
    )
)

avg_pheno_df = (
    all_pheno_df.lazy()
    .group_by(["annotation"])
    .agg(
        n_gene_trait_assoc = pl.col('region').n_unique(),
        mean_pheno = pl.col('mean_pheno_assoc').mean(),
        se_pheno = pl.col('mean_pheno_assoc').std() / pl.col('region').n_unique().sqrt(),
        # ci_low_pheno = pl.col('mean_pheno_assoc').quantile(0.025),
        # ci_high_pheno = pl.col('mean_pheno_assoc').quantile(0.975),
    )
    .with_columns(
        ci_low_pheno = pl.col('mean_pheno') - ci_factor*pl.col('se_pheno'),
        ci_high_pheno = pl.col('mean_pheno') + ci_factor*pl.col('se_pheno'),
    )

    .collect(engine='streaming')
)

avg_pheno_df

In [ ]:
plof_pheno_df = (
    avg_pheno_df
    .join(
        anno_config_df.drop('category').unique(), 
        on='annotation'
    )
    .with_columns(
        # n_label = pl.col('label') + "\n(n=" + pl.col('n_vars').cast(pl.Utf8) + ")"
        n_label = pl.col('label') + "\n(n=" + pl.col('n_gene_trait_assoc').cast(pl.Utf8) + ")"
    )
    .drop_nans()
)

# 2. Determine the categorical order for the labels
ordered_labels = (
    plof_pheno_df
    .sort("mean_pheno", descending=True)
    .select("n_label")
    .unique(maintain_order=True)
    .to_series()
)

# 3. Apply the ordering using pl.Enum
plof_pheno_df = plof_pheno_df.with_columns(pl.col("n_label").cast(pl.Enum(ordered_labels)))

# 4. Create the color dictionary (Polars style)
color_dict = dict(plof_pheno_df.select("annotation", "color").unique().iter_rows())

if exclude_clinvar:
    plot_title = f"{vc['x_label']} (no ClinVar)"
elif only_clinvar:
    plot_title = f"{vc['x_label']} (only ClinVar)"
else:
    plot_title = vc['x_label']

if only_snps:
    plot_title += " (SNPs)"

# Plot
(
    ggplot(
        plof_pheno_df,
        aes(x='n_label', y='mean_pheno')
    )
    # + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + geom_point(size=2)
    + geom_errorbar(
        aes(ymin='ci_low_pheno', ymax='ci_high_pheno'),
        width=0.2,
    )
    + labs(
        title=f"{plot_title} variants - (protein expression)",
        subtitle=f"Mean ± {ci_factor}×SEM across genes",
        x='', 
        y='Mean Olink z-score'
    )
    + coord_flip()
    + theme_minimal()
    + theme(
        figure_size=(7, plof_pheno_df['annotation'].n_unique()/2 + 0.5),
        axis_text=element_text(size=11),
        axis_title=element_text(size=13),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
# Calculate medians and join back
box_pheno_df = (
    all_pheno_df.collect()
    .join(
        anno_config_df.drop('category').unique(), 
        on='annotation'
    )
    .join(
        avg_pheno_df.select('annotation', 'n_gene_trait_assoc').unique(),
        on='annotation'
    )
    .with_columns(
        # n_label = pl.col('label') + "\n(n=" + pl.col('n_vars').cast(pl.Utf8) + ")"
        n_label = pl.col('label') + "\n(n=" + pl.col('n_gene_trait_assoc').cast(pl.Utf8) + ")"
    )
    .drop_nans()
    .with_columns(
        median_mean_pheno = pl.col('mean_pheno_assoc').median().over("annotation")
    )
)

# 2. Determine the categorical order for the labels
ordered_labels = (
    box_pheno_df
    .sort("median_mean_pheno", descending=True)
    .select("n_label")
    .unique(maintain_order=True)
    .to_series()
)

# 3. Apply the ordering using pl.Enum
box_pheno_df = box_pheno_df.with_columns(pl.col("n_label").cast(pl.Enum(ordered_labels)))

# 4. Create the color dictionary (Polars style)
color_dict = dict(box_pheno_df.select("annotation", "color").unique().iter_rows())

if exclude_clinvar:
    plot_title = f"{vc['x_label']} (no ClinVar)"
elif only_clinvar:
    plot_title = f"{vc['x_label']} (only ClinVar)"
else:
    plot_title = vc['x_label']

if only_snps:
    plot_title += " (SNPs)"

# Plot
(
    ggplot(
        box_pheno_df,
        aes(x='n_label', y='mean_pheno_assoc')
    )
    + geom_boxplot(width=0.75)
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + labs(
        title=f"{plot_title} variants - (protein expression)",
        x='', 
        y='Mean Olink z-score'
    )
    + coord_flip()
    + theme_minimal()
    + theme(
        figure_size=(7, box_pheno_df['annotation'].n_unique()/2 + 0.5),
        axis_text=element_text(size=12),
        axis_title=element_text(size=13),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)

# Check LOEUF scores of genes used by each category

In [ ]:
gnom = (
    pl.read_csv(
        "PATH_TO_FILE", 
        null_values=["NA"],
        separator='\t'
    )
    .filter(
        pl.col('transcript_type')=='protein_coding',
        pl.col('canonical')==True
        # pl.col('mane_select')==True
    )
    .select(['gene_id', 'lof.oe_ci.upper'])
    .rename({'gene_id': 'region', 'lof.oe_ci.upper': 'loeuf'})
    .unique()
    .drop_nulls()
)

gnom

In [ ]:
gnom_cats = (
    box_pheno_df[['annotation', 'n_label', 'region']].unique()
    .join(
        gnom,
        on='region',
        how='inner'
    )
)

gnom_cats

In [ ]:
(
    ggplot(
        gnom_cats.filter(pl.col('annotation').is_in(['clinvar_patho', 'clinvar_likely_patho'])),
        aes(fill='n_label', x='loeuf')
    )
    + geom_histogram(position='identity', alpha=0.3, bins=50)
    + geom_vline(xintercept=1, linetype='dashed', color='grey')
    + labs(
        title=f"{plot_title} variants - (protein expression)",
        x='LOEUF score'
    )
    + theme_minimal()
    + theme(
        figure_size=(7, 4),
        axis_text=element_text(size=12),
        axis_title=element_text(size=13),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
(
    ggplot(
        gnom_cats,
        aes(x='n_label', y='loeuf')
    )
    + geom_boxplot(width=0.75)
    + geom_hline(yintercept=1, linetype='dashed', color='grey')
    + labs(
        title=f"{plot_title} variants - (gene-trait assocs.)",
        x='', 
        y='LOEUF score'
    )
    + coord_flip()
    + theme_minimal()
    + theme(
        figure_size=(7, gnom_cats['annotation'].n_unique()/2 + 0.5),
        axis_text=element_text(size=12),
        axis_title=element_text(size=13),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)